<a href="https://colab.research.google.com/github/wingated/cs473/blob/main/labs/cs473_lab_week_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b>After clicking the "Open in Colab" link, copy the notebook to your own Google Drive before getting started, or it will not save your work</b></p>

# BYU CS 473 Lab Week 2

## Introduction:
Welcome to your first lab for CS 473, Advanced Machine Learning.

In machine learning, models often predict *unnormalized log probabilities*. These must often be converted into regular probabilities.

In this lab, you will explore the log-sum-exp function, which is described in the text (Sec. 2.5.4).  You will code up several variants of the function, and compare their performance.

# Part 1: Logsumexp
---
## Setup: The Iris Dataset
We'll begin by downloading the Iris dataset. The iris dataset is a simple, but very famous, dataset introduced to the world by RA Fisher (the “father” of modern statistics”) in 1939. The dataset has five columns:
* sepal length (cm)
* sepal width (cm)
* petal length (cm)
* petal width (cm)
* class

In order to get logits to play with, we'll first train a multinomial logistic regression model (Sec. 2.5.3).  This model naturally outputs logits.

In [105]:
import datasets
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

ds = datasets.load_dataset( "scikit-learn/iris" )

df = pd.DataFrame( ds['train'] )

X = np.array( df[['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']] )
Y = np.array( LabelEncoder().fit_transform( df['Species'] ) )

In [106]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression().fit(X,Y)

W = model.coef_
b = model.intercept_

b = np.reshape( b, (3,1))

logits = np.dot( W, X.T ) + b

---
## Exercise 1: convert logits to probabilities

Since our model outputs logits, they must be converted. To do this, we'll use the softmax function.

In [107]:
def softmax( logits ):
    # logits is a numpy matrix of d x N
    # where
    #   d is the number of classes
    #   N is the number of data points
    # use equation 2.99 (see also Eq. 2.94)

    # Exponentiate logits and apply Eq. 2.99.
    stable_logits = logits - np.max(logits, axis=0) # Subtract max (like class).
    exp_logits = np.exp(stable_logits)
    total = np.sum(exp_logits, axis=0)
    p_c = exp_logits / total

    return p_c

# **DOUBLE-CHECK OUTPUT WITH TA**

In [108]:
# print out test cases
probs = softmax(logits)
print(f'{probs[:, 0]}')
print(f'{probs[:, 120]}')

[9.81747643e-01 1.82523424e-02 1.43429404e-08]
[5.54234723e-06 2.39030160e-02 9.76091442e-01]


### test cases
probs = softmax( logits )
probs[:,0]
#### array([9.81803910e-01, 1.81960759e-02, 1.43430317e-08])
probs[:,120]
#### array([5.49519371e-06, 2.38812718e-02, 9.76113233e-01])

---
## Exercise 2: convert logits to probabilities

Now, code up the logsumexp function.  What test cases should you use for this function?

In [109]:
def logsumexp( logits ):
    # logits is a numpy matrix of d x N
    # where
    #   d is the number of classes
    #   N is the number of data points
    # use equation 2.100

    # Exponentiate logits and apply Eq. 2.100.
    m = np.max(logits, axis=0, keepdims=True)
    stable_logits = logits - m              # Subtract max (like class).
    exp_logits = np.exp(stable_logits)
    log_total = np.log(np.sum(exp_logits, axis=0, keepdims=True))

    return m + log_total

In [110]:
# test cases
probs = logsumexp(logits)
probs[:, 0]

array([7.36063027])

What should be printed??

# **(your answer here)**

`array([7.36063027])` should be printed.

---
## Exercise 3: explore underflow / overlow

First, code up a function that compares two distributions. This can be anything you want; you may consider things like the MSE.

In [111]:
def compare_probs( probs1, probs2 ):
    # Compute mean squared error (MSE).
    mse = np.mean((probs1 - probs2)**2)

    return mse

In [112]:
probs1 = softmax( logits )
# probs2 = logsumexp( logits )
probs2 = np.exp(logits - logsumexp(logits))
compare_probs( probs1, probs2 )

np.float64(1.62159731751564e-32)

Now, see what happens if you add (or subtract) a constant from logits. How big must the constant be before things start going haywire?

# **RESPONSE?? + DOUBLE-CHECK REQUIREMENTS WITH TA**

In [113]:
# your code here
og_probs = softmax(logits)
C_vals = [10, 100, 1000, 10_000, 100_000, 1_000_000, 10**8, 10**12, 10**16]
for C in C_vals:
    probs1 = softmax( logits + C )
    print(f'Comparison:  C = {C}, Diff:  {compare_probs(og_probs, probs1)}.')

# lse = logsumexp(logits + C)
# probs2 = np.exp((logits + C) - lse)

Comparison:  C = 10, Diff:  4.084907882116957e-33.
Comparison:  C = 100, Diff:  1.950857448826005e-31.
Comparison:  C = 1000, Diff:  1.4753166481656256e-29.
Comparison:  C = 10000, Diff:  4.895038131866129e-27.
Comparison:  C = 100000, Diff:  1.9438402513337563e-25.
Comparison:  C = 1000000, Diff:  1.698136147132689e-23.
Comparison:  C = 100000000, Diff:  2.576689341109173e-19.
Comparison:  C = 1000000000000, Diff:  2.3516041179511482e-11.
Comparison:  C = 10000000000000000, Diff:  0.0045431050570940105.


Now convert the logits to 16-bit precision, and re-run your experiments. Analyze the differences you see (2-3 sentences).

In [122]:
logits = logits.astype( np.float16 )

# your code here
og_probs = softmax(logits)
C_vals = [10, 100, 1000, 10_000, 100_000, 1_000_000, 10**8, 10**12, 10**16]
for C in C_vals:
    probs1 = softmax( logits + C )
    print(f'Comparison:  C = {C}, Diff:  {compare_probs(og_probs, probs1)}.')

Comparison:  C = 10, Diff:  1.1920928955078125e-07.
Comparison:  C = 100, Diff:  3.993511199951172e-06.
Comparison:  C = 1000, Diff:  0.00033283233642578125.
Comparison:  C = 10000, Diff:  0.055633544921875.
Comparison:  C = 100000, Diff:  nan.
Comparison:  C = 1000000, Diff:  nan.
Comparison:  C = 100000000, Diff:  nan.
Comparison:  C = 1000000000000, Diff:  nan.
Comparison:  C = 10000000000000000, Diff:  nan.


/tmp/ipykernel_3300/147017914.py:7: RuntimeWarning: overflow encountered in cast
  probs1 = softmax( logits + C )
/tmp/ipykernel_3300/3948559376.py:9: RuntimeWarning: invalid value encountered in subtract
  stable_logits = logits - np.max(logits, axis=0) # Subtract max (like class).


### Analysis

# **(Your analysis here)**

---
## Exercise 4: cleanly compute log probabilities

Sometimes, we want to compute log probabilities (which are different from logits), but we want to do so "cleanly", ie, while avoiding overflow / underflow. First, mathematically figure out what the log of the softmax is (ie, take the log of eq. 2.99), and then combine it with insights from coding up the logsumexp function. Hint: at the end of the day, you will simply shift each column by a per-column constant!

In [115]:
def log_logsumexp( logits ):
    # logits is a numpy matrix of d x N
    # where
    #   d is the number of classes
    #   N is the number of data points

    return logits - logsumexp(logits)

---
# Part 2: Probability Fundamentals

For the following exercises, you are encouraged to work both by hand and by code however makes the most sense.



## Exercise 1a: Joint Probability Distributions

You are given the following two binary variables, X and Y, that can each take on the values 0 or 1. Assuming X and Y are independent, calculate the joint probability table (2x2 table for P(X, Y)). Display as a numpy array.

P(X=0) = 0.6

P(X=1) = 0.4

P(Y=0) = 0.7

P(Y=1) = 0.3


In [116]:
# NOTE:  Independence implies P(X, Y) = P(X)P(Y).
# Calculate joint probability table and display.
PX_0, PX_1 = 0.6, 0.4
PY_0, PY_1 = 0.7, 0.3
joint_probs = np.array([
    [PX_0 * PY_0, PX_0 * PY_1],
    [PX_1 * PY_0, PX_1 * PY_1]
])

print(joint_probs)

[[0.42 0.18]
 [0.28 0.12]]


# **Next, compute the following conditional probabilities:**

P(X=0|Y=0) = ?

P(X=0|Y=1) = ?

P(Y=0|X=0) = ?

P(Y=0|X=1) = ?

In [117]:
# Store conditional probabilities (calculated by table above).
PX0_Y0 = joint_probs[0][0] / (joint_probs[0][0] + joint_probs[1][0])
PX0_Y1 = joint_probs[0][1] / (joint_probs[0][1] + joint_probs[1][1])
PY0_X0 = joint_probs[0][0] / (joint_probs[0][0] + joint_probs[0][1])
PY0_X1 = joint_probs[1][0] / (joint_probs[1][0] + joint_probs[1][1])

# Print out results.
print(f'P(X=0|Y=0) = {PX0_Y0}.')
print(f'P(X=0|Y=1) = {PX0_Y1}.')
print(f'P(Y=0|X=0) = {PY0_X0}.')
print(f'P(Y=0|X=1) = {PY0_X1}.')

P(X=0|Y=0) = 0.6.
P(X=0|Y=1) = 0.6.
P(Y=0|X=0) = 0.7.
P(Y=0|X=1) = 0.7.


Compare the result of these conditional probabilities to the original marginal probabilities given. What does this say about the relationship between variable dependence and using conditional probabilities? Write 1-2 sentences.

# **(Your answer here)**

As demonstrated above, `P(X=0|Y=0) = 0.6 = P(X=0|Y=1)` and `P(Y=0|X=0) = 0.7 = P(Y=0|X=1)` (which also implies `P(X=1|Y=0) = 1 - 0.6 = P(X=1|Y=1)` and `P(Y=1|X=0) = 1 - 0.7 = P(Y=1|X=1)`).  This indicates that the random variables `X` and `Y` are *independent* random variables because their probability is not influenced by our knowledge of the outcome of the other random variable.

## Exercise 1b: Joint Probability Distributions

*Now* consider this joint distribution:

|  | $Y = 0$ | $Y = 1$|
| :------- | :------: | -------: |
| $X = 0$  | 0.45  | 0.10  |
| $X = 1$  | 0.25  | 0.20  |

First, compute the marginals from the joint table

In [118]:
joint_prob_table = np.array([[0.45, 0.10],
                             [0.25, 0.20]])

# P(X=0) = ?
# P(X=1) = ?
# P(Y=0) = ?
# P(Y=1) = ?

# Your answer here
PX0 = joint_prob_table[0][0] + joint_prob_table[0][1]
PX1 = joint_prob_table[1][0] + joint_prob_table[1][1]
PY0 = joint_prob_table[0][0] + joint_prob_table[1][0]
PY1 = joint_prob_table[0][1] + joint_prob_table[1][1]

print(f'P(X=0) = {PX0}.')
print(f'P(X=1) = {PX1}.')
print(f'P(Y=0) = {PY0}.')
print(f'P(Y=1) = {PY1:.2f}.')

P(X=0) = 0.55.
P(X=1) = 0.45.
P(Y=0) = 0.7.
P(Y=1) = 0.30.


# **Compute the same conditional probabilities as above:**

P(X=0|Y=0) = ?

P(X=0|Y=1) = ?

P(Y=0|X=0) = ?

P(Y=0|X=1) = ?

In [123]:
# P(X=0|Y=0) = ?
# P(X=0|Y=1) = ?
# P(Y=0|X=0) = ?
# P(Y=0|X=1) = ?

# Your answer here
# Store conditional probabilities (calculated by table above).
PX0_Y0 = joint_prob_table[0][0] / (joint_prob_table[0][0] + joint_prob_table[1][0])
PX0_Y1 = joint_prob_table[0][1] / (joint_prob_table[0][1] + joint_prob_table[1][1])
PY0_X0 = joint_prob_table[0][0] / (joint_prob_table[0][0] + joint_prob_table[0][1])
PY0_X1 = joint_prob_table[1][0] / (joint_prob_table[1][0] + joint_prob_table[1][1])

# Print out results.
print(f'P(X=0|Y=0) = {PX0_Y0}.')
print(f'P(X=0|Y=1) = {PX0_Y1}.')
print(f'P(Y=0|X=0) = {PY0_X0}.')
print(f'P(Y=0|X=1) = {PY0_X1}.')

P(X=0|Y=0) = 0.6428571428571429.
P(X=0|Y=1) = 0.3333333333333333.
P(Y=0|X=0) = 0.8181818181818181.
P(Y=0|X=1) = 0.5555555555555556.


Check if the independence property $P(X, Y) = P(X)P(Y)$ holds for any cell.

In [120]:
# Compute P(X)P(Y) for each cell.
P_X0_Y0 = PX0 * PY0
P_X0_Y1 = PX0 * PY1
P_X1_Y0 = PX1 * PY0
P_X1_Y1 = PX1 * PY1

# Compute difference P(X, Y) - P(X)P(Y) for each cell.
X0_Y0_diff = joint_prob_table[0][0] - P_X0_Y0
X0_Y1_diff = joint_prob_table[0][1] - P_X0_Y1
X1_Y0_diff = joint_prob_table[1][0] - P_X1_Y0
X1_Y1_diff = joint_prob_table[1][1] - P_X1_Y1

# Check if independence property holds.
print(f'Check if P(X, Y) = P(X)P(Y) holds for any cell:')
print(f'P(X=0, Y=0) = P(X=0)P(Y=0):  {X0_Y0_diff == 0}.')
print(f'P(X=0, Y=1) = P(X=0)P(Y=1):  {X0_Y1_diff == 0}.')
print(f'P(X=1, Y=0) = P(X=1)P(Y=0):  {X1_Y0_diff == 0}.')
print(f'P(X=1, Y=1) = P(X=1)P(Y=1):  {X1_Y1_diff == 0}.')

if X0_Y0_diff == 0 or X0_Y1_diff == 0 or X1_Y0_diff == 0 or X1_Y1_diff == 0:
    print('The independence property holds for at least one cell.')
else:
    print('The independence property does NOT hold for any cell.')

Check if P(X, Y) = P(X)P(Y) holds for any cell:
P(X=0, Y=0) = P(X=0)P(Y=0):  False.
P(X=0, Y=1) = P(X=0)P(Y=1):  False.
P(X=1, Y=0) = P(X=1)P(Y=0):  False.
P(X=1, Y=1) = P(X=1)P(Y=1):  False.
The independence property does NOT hold for any cell.


Compare P(X=0|Y=0) to P(X=0|Y=1), and discuss what this says about the dependence between these variables (1-2 sentences).

# **(Your answer here)**

Because `P(X=0|Y=0)` $\neq$ `P(X=0|Y=1)`, we see that the conditional random variable `Y` influences the probability of `X`.  This clearly indicates that the random variables `X` and `Y` are dependent.

<br>

---
## Exercise 2: Bayes Theorem

After your yearly checkup, the doctor has bad news and good news. The bad news is that you tested positive
for a serious disease, and that the test is 99% accurate (i.e., the probability of testing positive given that you
have the disease is 0.99, as is the probability of testing negative given that you don’t have the disease). The
good news is that this is a rare disease, striking only one in 10,000 people. What are the chances that you
actually have the disease? (Show your calculations as well as giving the final result.)

*Hint: write out the variables you know, and think about what you'll need to calculate to find the final answer

In [121]:
# Define known values.
prob_pos_sick = 0.99
prob_neg_healthy = 0.99
prob_pos_healthy = 1 - prob_neg_healthy
prob_neg_sick = 1 - prob_pos_sick
prob_sick = 1 / 10_000
prob_healthy = 1 - prob_sick

# Apply Bayes' Rule.
prob_pos = (prob_pos_sick*prob_sick + prob_pos_healthy * prob_healthy)
prob_sick_pos = ((prob_pos_sick * prob_sick) / prob_pos)

print(f'The chance that one actually has the disease is:  {prob_sick_pos}.')

The chance that one actually has the disease is:  0.009803921568627442.
